# Experiment B: Extended Training / Overfitting Detection

## Purpose
Find when overfitting actually happens. Test if simple models overfit with extended training or if deeper models overfit faster.

## Setup
**Fixed settings:**
- Learning rate: 1e-3
- Batch size: 64
- min_df: 0.0005, max_df: 0.5

## Four conditions to test:

| Condition | Architecture | Epochs | Description |
|-----------|--------------|--------|----------|
| 1 | 300 → 128 → 2 | 50 | Simple model, long training |
| 2 | 300 → 128 → 2 | 100 | Simple model, very long training |
| 3 | 300 → 512 → 256 → 128 → 2 | 30 | Deep/wide model, medium training |
| 4 | 300 → 512 → 256 → 128 → 2 | 50 | Deep/wide model, long training |

## Setup (Run this once)
Load dataset and prepare preprocessing

In [1]:
# Tested for Python=3.10
import torch
from sklearn.feature_extraction.text import CountVectorizer
from datasets import load_from_disk, load_dataset
import gensim
import os
import pandas as pd
import numpy as np

# Load dataset
if os.path.exists("./imdb_dataset"):
    imdb = load_from_disk("./imdb_dataset")
else:
    imdb = load_dataset("imdb")
    imdb.save_to_disk("./imdb_dataset")

reviews_train = imdb['train']["text"]
reviews_test = imdb['test']["text"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

seeds = [42, 123, 456, 789, 1011]
print(f"Seeds to use: {seeds}")

Using cpu device
Seeds to use: [42, 123, 456, 789, 1011]


In [2]:
# Preprocess data (fixed for all conditions)
dictionary = CountVectorizer(min_df=0.0005, max_df=0.5).fit(reviews_train)
vocab_size = len(dictionary.vocabulary_)
print(f"Vocabulary size: {vocab_size}")

reviews_train_dict = dictionary.transform(reviews_train)
reviews_test_dict = dictionary.transform(reviews_test)

reviews_train_dt = torch.from_numpy(reviews_train_dict.todense())
reviews_test_dt = torch.from_numpy(reviews_test_dict.todense())

print(f"Train shape: {reviews_train_dt.shape}")
print(f"Test shape: {reviews_test_dt.shape}")

Vocabulary size: 15862
Train shape: torch.Size([25000, 15862])
Test shape: torch.Size([25000, 15862])


In [3]:
# Load Word2Vec embeddings
print("Loading Word2Vec model...")
model_w2v = gensim.models.KeyedVectors.load_word2vec_format('../.././GoogleNews-vectors-negative300.bin.gz', binary=True)
print("Word2Vec model loaded!")

word_embs_matrix = torch.zeros((vocab_size, 300), dtype=torch.float)
words_not_found = 0
for k, i in dictionary.vocabulary_.items():
    try:
        word_embs_matrix[i, :] = torch.tensor(model_w2v[k], dtype=torch.float)
    except KeyError:
        words_not_found += 1

print(f"Words not found in Word2Vec: {words_not_found}")

# Normalize and embed reviews
reviews_train_df_normalized = reviews_train_dt.float() / reviews_train_dt.float().sum(axis=1, keepdim=True)
reviews_test_df_normalized = reviews_test_dt.float() / reviews_test_dt.float().sum(axis=1, keepdim=True)
reviews_train_df_normalized[torch.isnan(reviews_train_df_normalized)] = 0
reviews_test_df_normalized[torch.isnan(reviews_test_df_normalized)] = 0

review_train_embs = reviews_train_df_normalized @ word_embs_matrix
review_test_embs = reviews_test_df_normalized @ word_embs_matrix

print(f"Review embeddings shape: {review_train_embs.shape}")

Loading Word2Vec model...
Word2Vec model loaded!
Words not found in Word2Vec: 1145
Review embeddings shape: torch.Size([25000, 300])


In [4]:
# Training and testing functions with loss tracking
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    train_loss = 0
    train_correct = 0
    total_samples = 0
    
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)
        train_loss += loss.item() * y.size(0)
        train_correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    return train_loss / total_samples, train_correct / total_samples

def test_loop(dataloader, model, loss_fn):
    model.eval()
    test_loss, correct = 0, 0
    total_samples = 0
    
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item() * y.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)
    
    return test_loss / total_samples, correct / total_samples

## Experiment B: Extended Training / Overfitting Detection

Testing 4 conditions with multiple seeds

In [5]:
# Define conditions
conditions = {
    1: {
        'name': 'Simple, 50 epochs',
        'model_fn': lambda: torch.nn.Sequential(
            torch.nn.Linear(300, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        ),
        'epochs': 50
    },
    2: {
        'name': 'Simple, 100 epochs',
        'model_fn': lambda: torch.nn.Sequential(
            torch.nn.Linear(300, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        ),
        'epochs': 100
    },
    3: {
        'name': 'Deep/Wide, 30 epochs',
        'model_fn': lambda: torch.nn.Sequential(
            torch.nn.Linear(300, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        ),
        'epochs': 30
    },
    4: {
        'name': 'Deep/Wide, 50 epochs',
        'model_fn': lambda: torch.nn.Sequential(
            torch.nn.Linear(300, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 256),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 2),
        ),
        'epochs': 50
    },
}

# Hyperparameters (fixed)
batch_size = 64
lr = 1e-3

# Prepare dataloaders
train_dataset = torch.utils.data.TensorDataset(
    review_train_embs, 
    torch.tensor(imdb['train']['label'], dtype=torch.long)
)
test_dataset = torch.utils.data.TensorDataset(
    review_test_embs, 
    torch.tensor(imdb['test']['label'], dtype=torch.long)
)

print("Dataloaders prepared.")
print(f"Conditions defined: {list(conditions.keys())}")

Dataloaders prepared.
Conditions defined: [1, 2, 3, 4]


In [6]:
# Run all conditions
all_results = []

for cond_id, cond_info in sorted(conditions.items()):
    cond_name = cond_info['name']
    epochs_to_train = cond_info['epochs']
    model_builder = cond_info['model_fn']
    
    print(f"\n{'='*80}")
    print(f"Condition {cond_id}: {cond_name}")
    print(f"{'='*80}")
    
    cond_results = {
        'best_test_accuracies': [],
        'best_epochs': [],
        'final_test_accuracies': []
    }
    
    for seed in seeds:
        print(f"\n  Seed {seed}:")
        torch.manual_seed(seed)
        
        # Create fresh dataloaders for this seed
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, num_workers=0
        )
        test_loader = torch.utils.data.DataLoader(
            test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
        )
        
        model = model_builder().to(device)
        loss_fn = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        best_test_acc = 0
        best_epoch_num = 0
        
        for epoch in range(epochs_to_train):
            train_loss, train_acc = train_loop(train_loader, model, loss_fn, optimizer)
            test_loss, test_acc = test_loop(test_loader, model, loss_fn)
            
            # Track best test accuracy
            if test_acc > best_test_acc:
                best_test_acc = test_acc
                best_epoch_num = epoch + 1
            
            # Record detailed results
            all_results.append({
                'condition': cond_id,
                'seed': seed,
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'train_accuracy': train_acc,
                'test_loss': test_loss,
                'test_accuracy': test_acc
            })
            
            # Print progress every 10 epochs
            if (epoch + 1) % 10 == 0 or epoch == epochs_to_train - 1:
                print(f"    Epoch {epoch+1:3d}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
                      f"Test Acc={test_acc:.4f}")
        
        cond_results['best_test_accuracies'].append(best_test_acc)
        cond_results['best_epochs'].append(best_epoch_num)
        cond_results['final_test_accuracies'].append(test_acc)
        
        overfitting_detected = "YES" if best_epoch_num < epochs_to_train else "NO"
        print(f"    Best Test Acc: {best_test_acc:.4f} @ epoch {best_epoch_num}")
        print(f"    Final Test Acc: {test_acc:.4f}")
        print(f"    Overfitting Detected: {overfitting_detected}")
    
    # Summary for this condition
    print(f"\n  CONDITION {cond_id} SUMMARY ({cond_name}):")
    print(f"    Best Test Accuracy:  {np.mean(cond_results['best_test_accuracies']):.4f} ± {np.std(cond_results['best_test_accuracies']):.4f}")
    print(f"    Best Epoch Reached:  {np.mean(cond_results['best_epochs']):.1f} ± {np.std(cond_results['best_epochs']):.1f}")
    print(f"    Final Test Accuracy: {np.mean(cond_results['final_test_accuracies']):.4f} ± {np.std(cond_results['final_test_accuracies']):.4f}")
    print(f"    Epochs Trained:      {epochs_to_train}")


Condition 1: Simple, 50 epochs

  Seed 42:
    Epoch  10: Train Loss=0.3244, Train Acc=0.8599, Test Acc=0.8606
    Epoch  20: Train Loss=0.3080, Train Acc=0.8684, Test Acc=0.8621
    Epoch  30: Train Loss=0.2914, Train Acc=0.8788, Test Acc=0.8613
    Epoch  40: Train Loss=0.2739, Train Acc=0.8863, Test Acc=0.8570
    Epoch  50: Train Loss=0.2584, Train Acc=0.8932, Test Acc=0.8571
    Best Test Acc: 0.8644 @ epoch 43
    Final Test Acc: 0.8571
    Overfitting Detected: YES

  Seed 123:
    Epoch  10: Train Loss=0.3294, Train Acc=0.8589, Test Acc=0.8536
    Epoch  20: Train Loss=0.3147, Train Acc=0.8670, Test Acc=0.8616
    Epoch  30: Train Loss=0.3042, Train Acc=0.8708, Test Acc=0.8622
    Epoch  40: Train Loss=0.2928, Train Acc=0.8772, Test Acc=0.8613
    Epoch  50: Train Loss=0.2823, Train Acc=0.8811, Test Acc=0.8636
    Best Test Acc: 0.8636 @ epoch 50
    Final Test Acc: 0.8636
    Overfitting Detected: NO

  Seed 456:
    Epoch  10: Train Loss=0.3285, Train Acc=0.8596, Test Acc=0.

In [7]:
# Create comprehensive results dataframe
results_df = pd.DataFrame(all_results)

print("\n" + "="*80)
print("DETAILED RESULTS: All epochs for all conditions and seeds")
print("="*80)
print(results_df.to_string(index=False))


DETAILED RESULTS: All epochs for all conditions and seeds
 condition  seed  epoch  train_loss  train_accuracy  test_loss  test_accuracy
         1    42      1    0.456953         0.79588   0.369155        0.84068
         1    42      2    0.358440         0.84540   0.348994        0.85004
         1    42      3    0.347319         0.85064   0.341131        0.85452
         1    42      4    0.341712         0.85264   0.351032        0.84876
         1    42      5    0.336212         0.85656   0.338577        0.85668
         1    42      6    0.333953         0.85696   0.333815        0.85692
         1    42      7    0.328991         0.85916   0.331081        0.85960
         1    42      8    0.328377         0.86104   0.332350        0.85828
         1    42      9    0.326378         0.86100   0.343388        0.84980
         1    42     10    0.324368         0.85992   0.328384        0.86056
         1    42     11    0.323430         0.86280   0.326108        0.86124
     

In [8]:
# Create summary table
summary_data = []

for cond_id in sorted(conditions.keys()):
    cond_name = conditions[cond_id]['name']
    cond_results = results_df[results_df['condition'] == cond_id]
    
    # Best test accuracy (max across all epochs)
    best_accs = cond_results.groupby('seed')['test_accuracy'].max()
    
    # Epoch where best accuracy occurs
    best_epochs = []
    for seed in seeds:
        seed_data = cond_results[cond_results['seed'] == seed]
        best_idx = seed_data['test_accuracy'].idxmax()
        best_ep = seed_data.loc[best_idx, 'epoch']
        best_epochs.append(best_ep)
    
    # Final epoch test accuracy
    final_accs = cond_results.groupby('seed')['test_accuracy'].last()
    
    summary_data.append({
        'Condition': cond_id,
        'Description': cond_name,
        'Best Test Acc (Mean)': f"{best_accs.mean():.4f}",
        'Best Test Acc (Std)': f"{best_accs.std():.4f}",
        'Best Epoch (Mean)': f"{np.mean(best_epochs):.1f}",
        'Best Epoch (Std)': f"{np.std(best_epochs):.1f}",
        'Final Test Acc (Mean)': f"{final_accs.mean():.4f}",
        'Final Test Acc (Std)': f"{final_accs.std():.4f}",
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("SUMMARY: Overfitting Detection Results")
print("="*80)
print(summary_df.to_string(index=False))

print("\n" + "="*80)
print("INTERPRETATION GUIDE")
print("="*80)
print("Best Epoch < Total Epochs => Overfitting likely (accuracy peaked, then declined or plateaued)")
print("Best Epoch ≈ Total Epochs => No overfitting (still improving at end of training)")
print("Final Acc << Best Acc => Strong overfitting signal")
print("Final Acc ≈ Best Acc => Model generalizes well")


SUMMARY: Overfitting Detection Results
 Condition          Description Best Test Acc (Mean) Best Test Acc (Std) Best Epoch (Mean) Best Epoch (Std) Final Test Acc (Mean) Final Test Acc (Std)
         1    Simple, 50 epochs               0.8639              0.0004              40.2              8.8                0.8600               0.0035
         2   Simple, 100 epochs               0.8639              0.0004              40.2              8.8                0.8543               0.0029
         3 Deep/Wide, 30 epochs               0.8622              0.0007              11.4              4.4                0.8506               0.0038
         4 Deep/Wide, 50 epochs               0.8622              0.0007              11.4              4.4                0.8412               0.0029

INTERPRETATION GUIDE
Best Epoch < Total Epochs => Overfitting likely (accuracy peaked, then declined or plateaued)
Best Epoch ≈ Total Epochs => No overfitting (still improving at end of training)
Final Ac

In [9]:
# Save results to CSV
results_df.to_csv('results_overfitting_detailed.csv', index=False)
summary_df.to_csv('results_overfitting_summary.csv', index=False)

print("Results saved to:")
print("  - results_overfitting_detailed.csv")
print("  - results_overfitting_summary.csv")

Results saved to:
  - results_overfitting_detailed.csv
  - results_overfitting_summary.csv
